In [8]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Tuple, Union

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.patches import Circle, FancyArrow, Rectangle
import matplotlib.patches as mpatches


def plot_availability_usage_4cat_map_filtered_by_capacity(
    *,
    availability_csv: Union[str, Path],
    usage_csv: Union[str, Path],
    capacity_csv: Union[str, Path],
    ny_tract_shp: Union[str, Path],
    output_dir: Union[str, Path],

    # ---- Column names ----
    csv_tract_col: str = "census_tract",
    shp_geoid_col: str = "GEOID",
    time_col: str = "time_slot",

    availability_value_col: str = "total_vehicle_available_norm",
    usage_value_col: str = "trips_starting_norm",  # or "trips_ending_norm"

    # ---- Dynamic value evaluation (tract × time -> tract) ----
    availability_agg_method: str = "mean",  # "mean","median","sum","max","min"
    usage_agg_method: str = "mean",

    # ---- Thresholds ----
    threshold_method: str = "quantile",  # "quantile" or "absolute"
    availability_threshold: float = 0.5,  # quantile (0-1) or absolute
    usage_threshold: float = 0.5,

    # ---- Capacity filtering ----
    nyc_borough_prefixes: Tuple[str, ...] = ("36061", "36047", "36081", "36005"),
    station_count_col: str = "num_station",
    min_stations: int = 1,
    gate_col: str = "total_capacity_norm",
    drop_zeros: bool = True,

    # ---- Plot ----
    title: str = "Availability × Usage Categorization (Filtered by Capacity)",
    figsize: Tuple[int, int] = (10, 10),
    edgecolor: str = "black",
    linewidth: float = 0.25,
    alpha: float = 0.92,
    add_basemap: bool = True,
    add_compass: bool = True,
    add_scalebar: bool = True,
    scalebar_segments_km: Tuple[int, int, int] = (0, 2, 5),
) -> Dict[str, Union[Path, float, str]]:
    """
    NYC tract map categorized into 4 groups based on tract-level dynamic availability and usage,
    filtered to tracts that have capacity (num_station >= min_stations, and total_capacity_norm > 0 if drop_zeros).

    Dynamic value evaluation:
      - availability_value per tract = aggregate over time (availability_agg_method)
      - usage_value per tract        = aggregate over time (usage_agg_method)
    """

    # ----------------------------
    # Helpers
    # ----------------------------
    def to_geoid11(s: pd.Series) -> pd.Series:
        return (
            s.astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
            .str.replace(r"\s+", "", regex=True)
            .str.zfill(11)
        )

    def agg_one_value_per_tract(df: pd.DataFrame, value_col: str, agg_method: str) -> pd.DataFrame:
        if csv_tract_col not in df.columns:
            raise KeyError(f"Missing tract column '{csv_tract_col}' in CSV.")
        if value_col not in df.columns:
            raise KeyError(f"Missing value column '{value_col}' in CSV.")

        out = df.copy()
        out[csv_tract_col] = to_geoid11(out[csv_tract_col])

        if time_col in out.columns:
            out[time_col] = pd.to_datetime(out[time_col], errors="coerce")

        g = out.groupby(csv_tract_col)[value_col]

        if agg_method == "mean":
            s = g.mean()
        elif agg_method == "median":
            s = g.median()
        elif agg_method == "sum":
            s = g.sum()
        elif agg_method == "max":
            s = g.max()
        elif agg_method == "min":
            s = g.min()
        else:
            raise ValueError("agg_method must be one of: mean, median, sum, max, min")

        return s.reset_index().rename(columns={value_col: f"{value_col}__{agg_method}"})

    def add_compass_rose(ax, xlim, ylim, frac_pos=(0.88, 0.88), size_frac=0.075):
        width = xlim[1] - xlim[0]
        height = ylim[1] - ylim[0]
        cx = xlim[0] + frac_pos[0] * width
        cy = ylim[0] + frac_pos[1] * height
        R = size_frac * min(width, height)

        ax.add_patch(Circle((cx, cy), R, facecolor="white", edgecolor="black", linewidth=1.1, zorder=10))
        ax.add_patch(Circle((cx, cy), R * 0.88, facecolor="white", edgecolor="black", linewidth=0.8, zorder=11))
        ax.text(cx, cy + R + 0.012 * height, "N", ha="center", va="bottom", fontsize=12, fontweight="bold", zorder=12)

        ax.add_patch(
            FancyArrow(
                cx, cy - 0.18 * R, 0, 0.72 * R,
                width=0.0,
                head_width=0.22 * R,
                head_length=0.28 * R,
                length_includes_head=True,
                facecolor="black",
                edgecolor="black",
                zorder=13,
            )
        )

    def add_scalebar_km(ax, xlim, ylim, frac_pos=(0.72, 0.06), bar_height_frac=0.012, segments_km=(0, 2, 5)):
        width = xlim[1] - xlim[0]
        height = ylim[1] - ylim[0]
        x0 = xlim[0] + frac_pos[0] * width
        y0 = ylim[0] + frac_pos[1] * height

        total_m = segments_km[-1] * 1000.0
        bar_h = bar_height_frac * height

        seg1_m = (segments_km[1] - segments_km[0]) * 1000.0
        seg2_m = (segments_km[2] - segments_km[1]) * 1000.0

        ax.add_patch(Rectangle((x0, y0), seg1_m, bar_h, facecolor="black", edgecolor="black", linewidth=0.8, zorder=10))
        ax.add_patch(Rectangle((x0 + seg1_m, y0), seg2_m, bar_h, facecolor="white", edgecolor="black", linewidth=0.8, zorder=10))

        ax.plot([x0, x0], [y0, y0 + bar_h], color="black", linewidth=1.0, zorder=11)
        ax.plot([x0 + seg1_m, x0 + seg1_m], [y0, y0 + bar_h], color="black", linewidth=1.0, zorder=11)
        ax.plot([x0 + total_m, x0 + total_m], [y0, y0 + bar_h], color="black", linewidth=1.0, zorder=11)

        ax.text(x0, y0 - 0.012 * height, f"{segments_km[0]}", ha="center", va="top", fontsize=10, zorder=12)
        ax.text(x0 + seg1_m, y0 - 0.012 * height, f"{segments_km[1]} km", ha="center", va="top", fontsize=10, zorder=12)
        ax.text(x0 + total_m, y0 - 0.012 * height, f"{segments_km[2]} km", ha="center", va="top", fontsize=10, zorder=12)
        ax.text(x0 + total_m / 2, y0 + bar_h + 0.008 * height, "Scale", ha="center", va="bottom", fontsize=10, zorder=12)

    # ----------------------------
    # Load + capacity universe filter
    # ----------------------------
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    cap = pd.read_csv(capacity_csv)
    for c in (csv_tract_col, station_count_col, gate_col):
        if c not in cap.columns:
            raise KeyError(f"capacity_csv missing '{c}'")

    cap = cap.copy()
    cap[csv_tract_col] = to_geoid11(cap[csv_tract_col])

    cap_agg = cap.groupby(csv_tract_col, as_index=False).agg(
        {station_count_col: "sum", gate_col: "mean"}
    )

    cap_universe = cap_agg.loc[cap_agg[station_count_col].fillna(0) >= min_stations, csv_tract_col].unique()
    if len(cap_universe) == 0:
        raise ValueError(f"0 tracts found with {station_count_col} >= {min_stations}")

    tracts = gpd.read_file(ny_tract_shp)
    if shp_geoid_col not in tracts.columns:
        raise KeyError(f"Tract shapefile missing '{shp_geoid_col}'")

    tracts = tracts.copy()
    tracts[shp_geoid_col] = to_geoid11(tracts[shp_geoid_col])

    gdf = tracts[tracts[shp_geoid_col].str.startswith(nyc_borough_prefixes)].copy()
    gdf = gdf[gdf[shp_geoid_col].isin(cap_universe)].copy()
    if gdf.empty:
        raise ValueError("After NYC + capacity filtering, 0 tracts remain. Check GEOID matching.")

    gdf = gdf.merge(cap_agg, left_on=shp_geoid_col, right_on=csv_tract_col, how="left")
    gdf = gdf.dropna(subset=[gate_col]).copy()
    if drop_zeros:
        gdf = gdf[gdf[gate_col] > 0].copy()
    if gdf.empty:
        raise ValueError("0 tracts remain after gate filtering. Try drop_zeros=False.")

    # ----------------------------
    # Aggregate dynamic values per tract
    # ----------------------------
    avail_df = pd.read_csv(availability_csv)
    usage_df = pd.read_csv(usage_csv)

    avail_agg_df = agg_one_value_per_tract(avail_df, availability_value_col, availability_agg_method)
    usage_agg_df = agg_one_value_per_tract(usage_df, usage_value_col, usage_agg_method)

    avail_col = f"{availability_value_col}__{availability_agg_method}"
    usage_col = f"{usage_value_col}__{usage_agg_method}"

    gdf = gdf.merge(avail_agg_df, on=csv_tract_col, how="left")
    gdf = gdf.merge(usage_agg_df, on=csv_tract_col, how="left")

    for c in (avail_col, usage_col):
        if c not in gdf.columns:
            raise KeyError(f"Expected column not found after merge: '{c}'")

    gdf = gdf.dropna(subset=[avail_col, usage_col]).copy()
    if gdf.empty:
        raise ValueError("After merging availability+usage, 0 tracts have both values non-null.")

    # ----------------------------
    # Thresholds
    # ----------------------------
    if threshold_method == "quantile":
        a_thr = float(gdf[avail_col].quantile(availability_threshold))
        u_thr = float(gdf[usage_col].quantile(usage_threshold))
    elif threshold_method == "absolute":
        a_thr = float(availability_threshold)
        u_thr = float(usage_threshold)
    else:
        raise ValueError("threshold_method must be 'quantile' or 'absolute'")

    # ----------------------------
    # 4-way categorization  ✅ FIXED
    # ----------------------------
    a_high = gdf[avail_col] >= a_thr
    u_high = gdf[usage_col] >= u_thr

    # Use default=0 so casting to int works in all environments
    gdf["category_code"] = np.select(
        [a_high & u_high, a_high & ~u_high, ~a_high & u_high, ~a_high & ~u_high],
        [1, 2, 3, 4],
        default=0,
    ).astype(int)

    # keep only valid 4 categories (drop category 0)
    gdf = gdf[gdf["category_code"].isin([1, 2, 3, 4])].copy()

    labels = {
        1: "High Availability + High Usage",
        2: "High Availability + Low Usage",
        3: "Low Availability + High Usage",
        4: "Low Availability + Low Usage",
    }
    gdf["category_label"] = gdf["category_code"].map(labels)

    # consistent palette (simple + readable)
    # Colorblind-friendly blue–orange palette
    color_map = {
    1: "#2166AC",  # High Availability + High Usage  (Dark Blue)
    2: "#67A9CF",  # High Availability + Low Usage   (Light Blue)
    3: "#FDAE61",  # Low Availability + High Usage   (Light Orange)
    4: "#D6604D",  # Low Availability + Low Usage    (Dark Orange)
    }

    # ----------------------------
    # Plot
    # ----------------------------
    gdf_web = gdf.to_crs(epsg=3857)
    xmin, ymin, xmax, ymax = gdf_web.total_bounds
    xlim, ylim = (xmin, xmax), (ymin, ymax)

    fig, ax = plt.subplots(figsize=figsize)
    gdf_web.plot(
        ax=ax,
        color=gdf_web["category_code"].map(color_map),
        edgecolor=edgecolor,
        linewidth=linewidth,
        alpha=alpha,
    )

    if add_basemap:
        ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)

    if add_compass:
        add_compass_rose(ax, xlim, ylim)

    if add_scalebar:
        add_scalebar_km(ax, xlim, ylim, segments_km=scalebar_segments_km)

    ax.set_title(
        f"{title}\n"
        f"Availability thr={a_thr:.6f} ({threshold_method}={availability_threshold}), "
        f"Usage thr={u_thr:.6f} ({threshold_method}={usage_threshold})",
        fontsize=13,
        fontweight="bold",
    )
    ax.axis("off")

    handles = [mpatches.Patch(color=color_map[k], label=labels[k]) for k in [1, 2, 3, 4]]
    ax.legend(handles=handles, loc="upper left", frameon=True, fontsize=10, title="Categories")

    outpath = output_dir / (
        f"NYC_AVAIL_USAGE_4CAT__A{availability_agg_method}_U{usage_agg_method}__{threshold_method}.png"
    )
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return {
        "png_path": outpath,
        "availability_threshold_value": a_thr,
        "usage_threshold_value": u_thr,
        "availability_col_used": avail_col,
        "usage_col_used": usage_col,
        "threshold_method": threshold_method,
    }


In [9]:
out = plot_availability_usage_4cat_map_filtered_by_capacity(
    availability_csv=r"D:\Research Fellowship\NYC_Docked_Output_v2\availability__norm__tract.csv",
    usage_csv=r"D:\Research Fellowship\NYC_Docked_Output_v2\usage_norm_hourly_tract.csv",
    capacity_csv=r"D:\Research Fellowship\NYC_Docked_Output_v2\capacity_tract_norm.csv",
    ny_tract_shp=r"D:\Research Fellowship\Summer Research Stuff\Clean_Utilities\Capacity\NYC\tl_2024_36_tract.shp",
    output_dir=r"D:\Research Fellowship\Outputs\MAPS",

    availability_value_col="total_vehicle_available_norm",
    usage_value_col="trips_starting_norm",   # or "trips_ending_norm"

    threshold_method="quantile",
    availability_threshold=0.5,
    usage_threshold=0.5,
)
print(out["png_path"])


D:\Research Fellowship\Outputs\MAPS\NYC_AVAIL_USAGE_4CAT__Amean_Umean__quantile.png
